In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key was found!')
elif not api_key.startswith('sk-proj-'):
    print('An API key was found, but it doesnt start with "sk-proj-", i.e, it is not an OpenAI API key')
elif api_key.strip() != api_key:
    print('An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them.')
else:
    print('The API key found and looks good so far!')


The API key found and looks good so far!


# LangChain

In [3]:
import os
from typing import Optional

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import Runnable, RunnableParallel, RunnablePassthrough

# --- Configuration ---
# Ensure your API key environment variable is set (e.g., OPENAI_API_KEY)
try:
    llm: Optional[ChatOpenAI] = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    if llm:
        print(f"Language model initialized: {llm.model_name}")
except Exception as e:
    print(f"Error initializing language model: {e}")
    llm = None


# --- Define Independent Chains ---
# These three chains represent distinct tasks that can be executed in parallel.

summarize_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Summarize the following topic concisely:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)

questions_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Generate three interesting questions about the following topic:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)

terms_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Identify 5-10 key terms from the following topic, separated by commas:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)


# --- Build the Parallel + Synthesis Chain ---

# 1. Define the block of tasks to run in parallel. The results of these,
#    along with the original topic, will be fed into the next step.
map_chain = RunnableParallel(
    {
        "summary": summarize_chain,
        "questions": questions_chain,
        "key_terms": terms_chain,
        "topic": RunnablePassthrough(),  # Pass the original topic through
    }
)

# 2. Define the final synthesis prompt which will combine the parallel results.
synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system", """Based on the following information:
     Summary: {summary}
     Related Questions: {questions}
     Key Terms: {key_terms}
     Synthesize a comprehensive answer."""),
    ("user", "Original topic: {topic}")
])

# 3. Construct the full chain by piping the parallel results directly
#    into the synthesis prompt, followed by the LLM and output parser.
full_parallel_chain = map_chain | synthesis_prompt | llm | StrOutputParser()


# --- Run the Chain ---
async def run_parallel_example(topic: str) -> None:
    """
    Asynchronously invokes the parallel processing chain with a specific topic
    and prints the synthesized result.

    Args:
        topic: The input topic to be processed by the LangChain chains.
    """
    if not llm:
        print("LLM not initialized. Cannot run example.")
        return

    print(f"\n--- Running Parallel LangChain Example for Topic: '{topic}' ---")
    try:
        # The input to `ainvoke` is the single 'topic' string, which is
        # then passed to each runnable in the `map_chain`.
        response = await full_parallel_chain.ainvoke(topic)
        print("\n--- Final Response ---")
        print(response)
    except Exception as e:
        print(f"\nAn error occurred during chain execution: {e}")

test_topic = "The history of space exploration"
# The notebook already runs an event loop, so await the coroutine directly.
await run_parallel_example(test_topic)

Language model initialized: gpt-4o-mini

--- Running Parallel LangChain Example for Topic: 'The history of space exploration' ---

--- Final Response ---
The history of space exploration is a remarkable journey that traces humanity’s quest to understand the cosmos, beginning with early astronomical observations and theories. The modern era of space exploration commenced with the launch of the Soviet satellite Sputnik in 1957, which not only marked the start of the space age but also ignited the Space Race between the United States and the Soviet Union. This competition significantly shaped technological advancements and fostered international collaborations in space exploration from the 1950s to the 1970s. The urgency to achieve milestones such as the first human in space—Yuri Gagarin's historic flight in 1961—and the first moon landing with Apollo 11 in 1969 led to rapid developments in aerospace technology, satellite communications, and scientific research capabilities.

Following th

# Google ADK

In [4]:
# Part of agent.py --> Follow https://google.github.io/adk-docs/get-started/quickstart/ to learn the setup
# Requires GOOGLE_API_KEY in .env (loaded above).

from google.adk.agents import LlmAgent, ParallelAgent, SequentialAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

GEMINI_MODEL = "gemini-flash-latest"

# --- 1. Define Researcher Sub-Agents (to run in parallel) ---

# Researcher 1: Renewable Energy
researcher_agent_1 = LlmAgent(
    name="RenewableEnergyResearcher",
    model=GEMINI_MODEL,
    instruction="""You are an AI Research Assistant specializing in energy.
Research the latest advancements in 'renewable energy sources'.
Use the Google Search tool provided.
Summarize your key findings concisely (1-2 sentences).
Output *only* the summary.
""",
    description="Researches renewable energy sources.",
    tools=[google_search],
    # Store result in state for the merger agent
    output_key="renewable_energy_result"
)

# Researcher 2: Electric Vehicles
researcher_agent_2 = LlmAgent(
    name="EVResearcher",
    model=GEMINI_MODEL,
    instruction="""You are an AI Research Assistant specializing in transportation.
Research the latest developments in 'electric vehicle technology'.
Use the Google Search tool provided.
Summarize your key findings concisely (1-2 sentences).
Output *only* the summary.
""",
    description="Researches electric vehicle technology.",
    tools=[google_search],
    # Store result in state for the merger agent
    output_key="ev_technology_result"
)

# Researcher 3: Carbon Capture
researcher_agent_3 = LlmAgent(
    name="CarbonCaptureResearcher",
    model=GEMINI_MODEL,
    instruction="""You are an AI Research Assistant specializing in climate solutions.
Research the current state of 'carbon capture methods'.
Use the Google Search tool provided.
Summarize your key findings concisely (1-2 sentences).
Output *only* the summary.
""",
    description="Researches carbon capture methods.",
    tools=[google_search],
    # Store result in state for the merger agent
    output_key="carbon_capture_result"
)

# --- 2. Create the ParallelAgent (Runs researchers concurrently) ---
# This agent orchestrates the concurrent execution of the researchers.
# It finishes once all researchers have completed and stored their results in state.
parallel_research_agent = ParallelAgent(
    name="ParallelWebResearchAgent",
    sub_agents=[researcher_agent_1, researcher_agent_2, researcher_agent_3],
    description="Runs multiple research agents in parallel to gather information."
)

# --- 3. Define the Merger Agent (Runs *after* the parallel agents) ---
# This agent takes the results stored in the session state by the parallel agents
# and synthesizes them into a single, structured response with attributions.
merger_agent = LlmAgent(
    name="SynthesisAgent",
    model=GEMINI_MODEL,  # Or potentially a more powerful model if needed for synthesis
    instruction="""You are an AI Assistant responsible for combining research findings into a structured report.

Your primary task is to synthesize the following research summaries, clearly attributing findings to their source areas. Structure your response using headings for each topic. Ensure the report is coherent and integrates the key points smoothly.

**Crucially: Your entire response MUST be grounded *exclusively* on the information provided in the 'Input Summaries' below. Do NOT add any external knowledge, facts, or details not present in these specific summaries.**

**Input Summaries:**

*   **Renewable Energy:**
    {renewable_energy_result}

*   **Electric Vehicles:**
    {ev_technology_result}

*   **Carbon Capture:**
    {carbon_capture_result}

**Output Format:**

## Summary of Recent Sustainable Technology Advancements

### Renewable Energy Findings
(Based on RenewableEnergyResearcher's findings)
[Synthesize and elaborate *only* on the renewable energy input summary provided above.]

### Electric Vehicle Findings
(Based on EVResearcher's findings)
[Synthesize and elaborate *only* on the EV input summary provided above.]

### Carbon Capture Findings
(Based on CarbonCaptureResearcher's findings)
[Synthesize and elaborate *only* on the carbon capture input summary provided above.]

### Overall Conclusion
[Provide a brief (1-2 sentence) concluding statement that connects *only* the findings presented above.]

Output *only* the structured report following this format. Do not include introductory or concluding phrases outside this structure, and strictly adhere to using only the provided input summary content.
""",
    description="Combines research findings from parallel agents into a structured, cited report, strictly grounded on provided inputs.",
    # No tools needed for merging
    # No output_key needed here, as its direct response is the final output of the sequence
)


# --- 4. Create the SequentialAgent (Orchestrates the overall flow) ---
# This is the main agent that will be run. It first executes the ParallelAgent
# to populate the state, and then executes the MergerAgent to produce the final output.
sequential_pipeline_agent = SequentialAgent(
    name="ResearchAndSynthesisPipeline",
    # Run parallel research first, then merge
    sub_agents=[parallel_research_agent, merger_agent],
    description="Coordinates parallel research and synthesizes the results."
)

root_agent = sequential_pipeline_agent

# --- 5. Run the pipeline ---
APP_NAME = "parallel_research_app"
USER_ID = "user_1"

runner = InMemoryRunner(agent=root_agent, app_name=APP_NAME)
session = await runner.session_service.create_session(app_name=APP_NAME, user_id=USER_ID)

message = types.Content(
    role="user",
    parts=[types.Part(text="Research the latest sustainable technology advancements.")],
)

async for event in runner.run_async(
    user_id=USER_ID, session_id=session.id, new_message=message
):
    if event.is_final_response() and event.content and event.content.parts:
        print(f"--- {event.author} ---")
        print(event.content.parts[0].text)
        print()


/var/folders/_n/81b2c8ld4ybgj_p8m3_g6rc00000gn/T/ipykernel_32526/1525665019.py:64: DeprecationWarning: ParallelAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  parallel_research_agent = ParallelAgent(
/var/folders/_n/81b2c8ld4ybgj_p8m3_g6rc00000gn/T/ipykernel_32526/1525665019.py:123: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  sequential_pipeline_agent = SequentialAgent(
Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


--- EVResearcher ---
Recent advancements in electric vehicle technology are centered on breakthroughs in solid-state and sodium-ion batteries that offer higher energy density, faster charging speeds, and improved thermal safety. Simultaneously, the integration of ultra-fast charging infrastructure and bidirectional vehicle-to-grid (V2G) systems is transforming EVs into efficient, grid-supporting energy assets.

--- CarbonCaptureResearcher ---
Recent advancements in carbon capture focus on scaling both point-source industrial capture and direct air capture (DAC) through innovative materials, including metal-organic frameworks (MOFs), novel solvent systems, and advanced membrane separation technologies. These developments aim to significantly reduce the energy intensity and operational costs of capture while integrating with expanding geological storage networks for permanent sequestration.

--- RenewableEnergyResearcher ---
Recent advancements in renewable energy include high-efficiency

In [5]:
# Same pipeline, written with the graph-based Workflow API (google-adk 2.x).
# ParallelAgent / SequentialAgent are deprecated in favor of Workflow.
# Requires GOOGLE_API_KEY in .env (loaded above).

from google.adk import Agent, Workflow
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.adk.workflow import JoinNode
from google.genai import types

GEMINI_MODEL = "gemini-flash-latest"

# --- 1. Define Researcher Agents (nodes that run in parallel) ---

researcher_agent_1 = Agent(
    name="RenewableEnergyResearcher",
    model=GEMINI_MODEL,
    instruction="""You are an AI Research Assistant specializing in energy.
Research the latest advancements in 'renewable energy sources'.
Use the Google Search tool provided.
Summarize your key findings concisely (1-2 sentences).
Output *only* the summary.
""",
    description="Researches renewable energy sources.",
    tools=[google_search],
)

researcher_agent_2 = Agent(
    name="EVResearcher",
    model=GEMINI_MODEL,
    instruction="""You are an AI Research Assistant specializing in transportation.
Research the latest developments in 'electric vehicle technology'.
Use the Google Search tool provided.
Summarize your key findings concisely (1-2 sentences).
Output *only* the summary.
""",
    description="Researches electric vehicle technology.",
    tools=[google_search],
)

researcher_agent_3 = Agent(
    name="CarbonCaptureResearcher",
    model=GEMINI_MODEL,
    instruction="""You are an AI Research Assistant specializing in climate solutions.
Research the current state of 'carbon capture methods'.
Use the Google Search tool provided.
Summarize your key findings concisely (1-2 sentences).
Output *only* the summary.
""",
    description="Researches carbon capture methods.",
    tools=[google_search],
)

# --- 2. Join Node (waits for all three researchers, then fans their outputs in) ---
# Its output is a dict keyed by the name of each upstream node.
gather_research = JoinNode(name="GatherResearch")

# --- 3. Function Node (formats the joined outputs for the merger) ---
TOPICS = {
    "RenewableEnergyResearcher": "Renewable Energy",
    "EVResearcher": "Electric Vehicles",
    "CarbonCaptureResearcher": "Carbon Capture",
}


def format_summaries(node_input: dict[str, str]) -> str:
    """Turn the joined researcher outputs into a labeled 'Input Summaries' block."""
    return "\n\n".join(
        f"*   **{topic}:**\n    {node_input[name]}" for name, topic in TOPICS.items()
    )


# --- 4. Define the Merger Agent (the final node of the graph) ---
merger_agent = Agent(
    name="SynthesisAgent",
    model=GEMINI_MODEL,  # Or potentially a more powerful model if needed for synthesis
    instruction="""You are an AI Assistant responsible for combining research findings into a structured report.

Your primary task is to synthesize the research summaries you receive as input, clearly attributing findings to their source areas. Structure your response using headings for each topic. Ensure the report is coherent and integrates the key points smoothly.

**Crucially: Your entire response MUST be grounded *exclusively* on the information provided in the input summaries. Do NOT add any external knowledge, facts, or details not present in these specific summaries.**

**Output Format:**

## Summary of Recent Sustainable Technology Advancements

### Renewable Energy Findings
(Based on RenewableEnergyResearcher's findings)
[Synthesize and elaborate *only* on the renewable energy input summary provided above.]

### Electric Vehicle Findings
(Based on EVResearcher's findings)
[Synthesize and elaborate *only* on the EV input summary provided above.]

### Carbon Capture Findings
(Based on CarbonCaptureResearcher's findings)
[Synthesize and elaborate *only* on the carbon capture input summary provided above.]

### Overall Conclusion
[Provide a brief (1-2 sentence) concluding statement that connects *only* the findings presented above.]

Output *only* the structured report following this format. Do not include introductory or concluding phrases outside this structure, and strictly adhere to using only the provided input summary content.
""",
    description="Combines research findings from the parallel nodes into a structured, cited report, strictly grounded on provided inputs.",
)

# --- 5. Build the Workflow graph ---
# START fans out to the three researchers, which fan back in through the join
# node; the joined result is formatted and handed to the merger agent.
root_agent = Workflow(
    name="ResearchAndSynthesisPipeline",
    edges=[
        (
            "START",
            (researcher_agent_1, researcher_agent_2, researcher_agent_3),
            gather_research,
        ),
        (gather_research, format_summaries, merger_agent),
    ],
)

# --- 6. Run the workflow ---
APP_NAME = "workflow_research_app"
USER_ID = "user_1"

runner = InMemoryRunner(agent=root_agent, app_name=APP_NAME)
session = await runner.session_service.create_session(app_name=APP_NAME, user_id=USER_ID)

message = types.Content(
    role="user",
    parts=[types.Part(text="Research the latest sustainable technology advancements.")],
)

async for event in runner.run_async(
    user_id=USER_ID, session_id=session.id, new_message=message
):
    if event.is_final_response() and event.content and event.content.parts:
        text = event.content.parts[0].text
        if text:
            print(f"--- {event.author} ---")
            print(text)
            print()


--- RenewableEnergyResearcher ---
Recent advancements in renewable energy are led by the commercialization of ultra-high-efficiency perovskite tandem solar cells, expanding agrivoltaic and floating solar systems, and cheaper green hydrogen production. Concurrently, breakthroughs in long-duration storage technologies like iron-air batteries and AI-driven smart grids are significantly enhancing grid flexibility and overcoming intermittency challenges.

--- CarbonCaptureResearcher ---
Current carbon capture technologies are rapidly expanding from established point-source industrial solvent systems toward direct air capture (DAC) and mineralization, driven by advancements in metal-organic frameworks and solid sorbents that improve capture efficiency. However, commercial deployment at a climate-relevant scale remains constrained by high capital costs, energy intensity, and the need for expanded transport and permanent geological storage infrastructure.

--- EVResearcher ---
Recent advanceme